In [1]:
import numpy as np
import pandas as pd
from datetime import datetime
import os

# for visualization
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='ticks', font_scale=1.5)


import warnings
warnings.filterwarnings('ignore')


In [2]:
%pip install 'bertopic>=0.17,<0.18' 'sentence-transformers>=3,<4' 'transformers>=4.41,<5' 'umap-learn>=0.5,<0.6' 'hdbscan>=0.8.33' spacy adjustText ipykernel


  Using cached bertopic-0.17.4-py3-none-any.whl.metadata (24 kB)
  Using cached sentence_transformers-3.4.1-py3-none-any.whl.metadata (10 kB)
  Using cached transformers-4.57.6-py3-none-any.whl.metadata (43 kB)
  Using cached umap_learn-0.5.12-py3-none-any.whl.metadata (24 kB)
  Using cached hdbscan-0.8.44-cp314-cp314-macosx_10_15_universal2.whl.metadata (15 kB)
  Using cached adjusttext-1.4.0-py3-none-any.whl.metadata (3.4 kB)
  Using cached llvmlite-0.49.0.tar.gz (194 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of sentence-transformers to determine which version is compatible with other requirements. This could take a while.
  Using cached sentence_transformers-3.4.0-py3-none-any.whl.metadata (10 kB)
  Using cached sentence_transformers-3.3.1-py3-none-any.whl.metadata (10 kB)
  Using cached sentence_transformers-3.3.0-py3-none-any.whl.metadata (10 k

In [3]:
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired
from sentence_transformers import SentenceTransformer
from umap import UMAP
from tqdm import tqdm


ModuleNotFoundError: No module named 'bertopic'

In [ ]:
INPUT_CSV = 'earnings_calls.csv'

df = pd.read_csv(INPUT_CSV)
df.head()


In [ ]:
df.info()

In [ ]:
sample_idx = min(10, len(df) - 1)
print(df.iloc[sample_idx, 0])


In [ ]:
# Clean exchange labels like 'NASDAQ: BILI' -> 'NASDAQ'.
df['exchange'] = df['exchange'].str.split(':').str[0]

# Clean Motley Fool date strings such as 'Aug 27, 2020, 9:00 p.m. ET'.
target_words = ['Call', 'Transcript']
df['date'] = (
    df['date']
    .astype('string')
    .str.replace(r'^.*?(?:' + '|'.join(target_words) + r')\s*', '', regex=True)
    .str.replace('a.m.', 'AM', regex=False)
    .str.replace('p.m.', 'PM', regex=False)
    .str.replace(r'\s+ET$', '', regex=True)
)
df['date'] = pd.to_datetime(df['date'], format='mixed', errors='coerce')

df.head()


In [ ]:
df['exchange'].value_counts().sort_index(ascending=False)



In [ ]:
# replace text in the exchange column with the first letter of the text
df['exchange'] = df['exchange'].str.replace('NYSEMKT', 'NYSE', regex=False)
df['exchange'] = df['exchange'].str.replace('(NASDAQ', 'NASDAQ', regex=False)
df['exchange'] = df['exchange'].str.replace('(NYSE', 'NYSE', regex=False)

df['exchange'].value_counts().sort_index(ascending=False)

In [ ]:
df.isnull().sum()

In [ ]:
df['date'] = df['date'].ffill()

In [ ]:
print("DATASET OVERVIEW:")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
print(f"Unique companies: {df['ticker'].nunique()}")
print(f"Unique exchanges: {df['exchange'].unique()}")
print(f"Total earnings calls: {len(df)}")

In [ ]:
# Distribution by year and quarter
df['year'] = pd.to_datetime(df['date']).dt.year

In [ ]:
# get the quarter from the splitting the q column 
df['quarter'] = df['q'].str.split('-').str[-1]

In [ ]:
print("\nDISTRIBUTION BY YEAR:")
year_counts = df['year'].value_counts().sort_index()
print(year_counts)

print("\nDISTRIBUTION BY QUARTER:")
quarter_counts = df['quarter'].value_counts().sort_index()
print(quarter_counts)

In [ ]:
# Top companies by number of earnings calls
print("\nTOP 10 COMPANIES BY EARNINGS CALLS:")
top_companies = df['ticker'].value_counts().head(10)
print(top_companies)

In [ ]:
# Sample transcript length analysis
df['transcript_length'] = df['transcript'].str.len()
print(f"\nTRANSCRIPT LENGTH STATS:")
print(f"Mean length: {df['transcript_length'].mean():.0f} characters")
print(f"Median length: {df['transcript_length'].median():.0f} characters")
print(f"Min length: {df['transcript_length'].min():.0f} characters")
print(f"Max length: {df['transcript_length'].max():.0f} characters")

In [ ]:
# Show sample transcript structure
print("\nSAMPLE TRANSCRIPT PREVIEW:")
sample_transcript = df.iloc[1]['transcript']
print(f"Company: {df.iloc[1]['ticker']}")
print(f"Date: {df.iloc[1]['date']}")
print(f"First 1000 characters:\n{sample_transcript}...")

In [ ]:
# Analyze data quality issues that might affect BERTopic analysis
print("DATA QUALITY ANALYSIS:")
print("="*30)

# Check for very short transcripts (might be incomplete)
short_transcripts = df[df['transcript_length'] < 5000]
print(f"Transcripts < 5000 chars: {len(short_transcripts)} ({len(short_transcripts)/len(df)*100:.1f}%)")

In [ ]:
# Check for duplicate entries
duplicates = df.duplicated(subset=['ticker', 'date']).sum()
print(f"Duplicate ticker-date combinations: {duplicates}")

In [ ]:
# Check date format consistency
try:
    df['parsed_date'] = pd.to_datetime(df['date'])
    print("Date parsing: SUCCESS")
except:
    print("Date parsing: ISSUES DETECTED")


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
# Earnings calls by year
year_counts = df['year'].value_counts().sort_index()
axes[0,0].bar(year_counts.index, year_counts.values)
axes[0,0].set_title('Earnings Calls by Year')
axes[0,0].set_xlabel('Year')
axes[0,0].set_ylabel('Number of Calls')

# Earnings calls by quarter
quarter_counts = df['quarter'].value_counts().sort_index()
axes[0,1].bar(quarter_counts.index, quarter_counts.values)
axes[0,1].set_title('Earnings Calls by Quarter')
axes[0,1].set_xlabel('Quarter')
axes[0,1].set_ylabel('Number of Calls')

# Transcript length distribution
axes[1,0].hist(df['transcript_length'], bins=50, alpha=0.7)
axes[1,0].set_title('Distribution of Transcript Lengths')
axes[1,0].set_xlabel('Characters')
axes[1,0].set_ylabel('Frequency')

# Top companies
top_10 = df['ticker'].value_counts().head(10)
axes[1,1].barh(range(len(top_10)), top_10.values)
axes[1,1].set_yticks(range(len(top_10)))
axes[1,1].set_yticklabels(top_10.index)
axes[1,1].set_title('Top 10 Companies by Earnings Calls')
axes[1,1].set_xlabel('Number of Calls')

plt.tight_layout()
plt.show()

In [ ]:
import re
import spacy
from typing import List

# Load spaCy model if available; fall back to a blank English sentencizer.
try:
    nlp = spacy.load("en_core_web_sm", disable=["ner", "parser"])
except OSError:
    nlp = spacy.blank("en")
if "sentencizer" not in nlp.pipe_names:
    nlp.add_pipe("sentencizer")
# -----------------------------
# 1. Domain-specific stopwords
# -----------------------------
FIN_STOPWORDS = {
    "operator", "thank", "thanks", "quarter", "fiscal",
    "year", "company", "corp", "corporation",
    "earnings", "call", "session",
    "senior vice president", "chief financial officer", "chairman of the board", "chief executive officer",
    "Senior Vice President", "Chief Financial Officer", "Chairman of the Board", "Chief Executive Officer"
}

# Keep negations & modal verbs
NEGATIONS = {"not", "no", "never", "none"}
MODALS = {"may", "might", "could", "should", "would"}

# -----------------------------
# 2. Boilerplate removal
# -----------------------------

def remove_boilerplate(text: str) -> str:
    patterns = [
        r"forward-looking statements.*?\.\s*",
        r"good day.*?earnings call\.\s*",
        r"we will now begin the question-and-answer session\.\s*",
    ]
    for p in patterns:
        text = re.sub(p, " ", text, flags=re.IGNORECASE | re.DOTALL)

    # Normalize whitespace (including newlines/tabs) to avoid glue or weird gaps
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text) 
    return text.strip()

def remove_speaker_labels(text: str) -> str:
    return re.sub(r"^(?:[A-Za-z]+(?:[ \t]+[A-Za-z]+){0,5})(?:\s*[)]∗[^)]*)?:\s*", "", text, flags=re.MULTILINE)

# -----------------------------
# 4. Sentence-level preprocessing
# -----------------------------
def preprocess_sentences(text: str) -> List[str]:
    doc = nlp(text)
    processed = []

    for sent in doc.sents:
        tokens = []
        for token in sent:
            if token.is_punct or token.is_space:
                continue

            lemma = token.lemma_.lower()

            if lemma in FIN_STOPWORDS and lemma not in NEGATIONS:
                continue

            if lemma in NEGATIONS or lemma in MODALS:
                tokens.append(lemma)
            elif token.is_stop:
                continue
            else:
                tokens.append(lemma)

        if len(tokens) > 3:
            processed.append(" ".join(tokens))

    return processed

# -----------------------------
# 5. Full preprocessing pipeline
# -----------------------------
def preprocess_transcript(raw_text: str) -> List[str]:
    text = raw_text.lower()
    text = remove_boilerplate(text)
    text = remove_speaker_labels(text)
    sentences = preprocess_sentences(text)
    return sentences


In [ ]:
def split_by_speaker(raw_text: str) -> List[str]:

    text = raw_text.lower()

    
    transcript = remove_boilerplate(text)

    transcript = remove_speaker_labels(transcript)

    titles = [
    'Chief administrative officer',
    'Chief analytics officer',
    'Chief brand officer',
    'Chief business development officer',
    'Chief business officer',
    'Chief commercial officer',
    'Chief communications officer',
    'Chief compliance officer',
    'Chief confluence officer',
    'Chief content officer',
    'Chief creative officer',
    'Chief customer officer',
    'Chief data officer',
    'Chief design officer',
    'Chief digital officer',
    'Chief diversity officer',
    'Chief executive officer',
    'Chief experience officer',
    'Chief financial officer',
    'Chief gaming officer',
    'Chief genealogical officer',
    'Chief growth officer',
    'Chief human resources officer',
    'Chief information officer',
    'Chief information officer (higher education)',
    'Chief information security officer',
    'Chief innovation officer',
    'Chief investment officer',
    'Chief knowledge officer',
    'Chief learning officer',
    'Chief legal officer',
    'Chief marketing officer',
    'Chief operations officer',
    'Chief people officer',
    'Chief privacy officer',
    'Chief process officer',
    'Chief product officer',
    'Chief quality officer',
    'Chief reputation officer',
    'Chief research officer',
    'Chief restructuring officer',
    'Chief risk officer',
    'Chief scientific officer',
    'Chief security officer',
    'Chief services officer',
    'Chief solutions officer',
    'Chief strategy officer',
    'Chief supply chain officer',
    'Chief sustainability officer',
    'Chief technical officer',
    'Chief technology officer',
    'Chief technology security officer',
    'Chief visibility officer',
    'Chief visionary officer',
    'Chief web officer',
    'President',
    'Operator']
    
    delimiters = [i.lower() + '\n' for i in titles]

    # Build regex pattern
    pattern = r"|".join(map(re.escape, delimiters))
    
    parts = re.split(pattern, transcript.lower())
    
    # Clean result
    split_by_speaker_temp = [p.strip() for p in parts if p.strip()]

    split_by_speaker_temp_2 = []

    for piece in split_by_speaker_temp:
        
        doc = nlp(piece)
    
        processed = []
    
        for sent in doc.sents:
            tokens = []
            for token in sent:
                if token.is_punct or token.is_space:
                    continue
    
                lemma = token.lemma_.lower()
    
                if lemma in FIN_STOPWORDS and lemma not in NEGATIONS:
                    continue
    
                if lemma in NEGATIONS or lemma in MODALS:
                    tokens.append(lemma)
                elif token.is_stop:
                    continue
                else:
                    tokens.append(lemma)
    
            if len(tokens) > 3:
                processed.append(" ".join(tokens))
            
        split_by_speaker_temp_2.append("".join(processed))

    split_by_speaker = split_by_speaker_temp_2
    
    return split_by_speaker

In [ ]:
# Show sample transcript structure
print("\nSAMPLE TRANSCRIPT PREVIEW:")
sample_transcript = df.iloc[1]['transcript']
print(f"Company: {df.iloc[1]['ticker']}")
print(f"Date: {df.iloc[1]['date']}")
print(f"First 1000 characters:\n{sample_transcript}...")

In [ ]:
stat = [len(split_by_speaker(df.iloc[i]['transcript'])) for i in tqdm(range(30))]

In [ ]:
doc = split_by_speaker(df.iloc[1]['transcript'])

In [ ]:
docs = df['transcript'].dropna().astype(str).tolist()

# Optional: save chunked transcripts for external reuse.
with open('docs.txt', 'w', encoding='utf-8') as file:
    file.write('\n\n'.join(docs))

representation_model = KeyBERTInspired()
sentence_model = SentenceTransformer('all-MiniLM-L6-v2')

embedding_cache = 'sentence_embeddings.npy'
if os.path.exists(embedding_cache):
    sentence_embeddings = np.load(embedding_cache)
    print(f'Loaded cached embeddings from {embedding_cache}')
else:
    sentence_embeddings = sentence_model.encode(docs, show_progress_bar=True)
    np.save(embedding_cache, sentence_embeddings)
    print(f'Saved embeddings to {embedding_cache}')

print('Fitting the BERTopic model')
topic_model = BERTopic(
    embedding_model=sentence_model,
    representation_model=representation_model,
    min_topic_size=5,
    calculate_probabilities=False,
    verbose=True,
)
topics, probs = topic_model.fit_transform(docs, embeddings=sentence_embeddings)
print('Finished fitting the BERTopic model')


In [ ]:
topic_model.get_topic_info()


In [ ]:
topic_model.get_document_info(docs).head()

In [ ]:
topic_model.save('topic_model_earnings_calls', serialization='pickle')

In [ ]:
import itertools
import pandas as pd

reduced_embeddings_2d = UMAP(
    n_neighbors=10,
    n_components=2,
    min_dist=0.0,
    metric='cosine',
    random_state=42,
).fit_transform(sentence_embeddings)

colors = itertools.cycle(['#e6194b', '#3cb44b', '#ffe119', '#4363d8', '#f58231', '#911eb4', '#46f0f0', '#f032e6', '#bcf60c', '#fabebe', '#008080', '#e6beff', '#9a6324', '#fffac8', '#800000', '#aaffc3', '#808000', '#ffd8b1', '#000075', '#808080', '#ffffff', '#000000'])
color_key = {str(topic): next(colors) for topic in set(topic_model.topics_) if topic != -1}

plot_df = pd.DataFrame({
    'x': reduced_embeddings_2d[:, 0],
    'y': reduced_embeddings_2d[:, 1],
    'Topic': [str(t) for t in topic_model.topics_],
})
plot_df['Length'] = [len(doc) for doc in docs]
plot_df = plot_df.loc[plot_df.Topic != '-1'].copy()
plot_df = plot_df.loc[(plot_df.y > -10) & (plot_df.y < 10) & (plot_df.x < 10) & (plot_df.x > -10), :]
plot_df['Topic'] = plot_df['Topic'].astype('category')

mean_df = plot_df.groupby('Topic', observed=True).mean(numeric_only=True).reset_index()
mean_df.Topic = mean_df.Topic.astype(int)
mean_df = mean_df.sort_values('Topic')


In [ ]:
import seaborn as sns
from matplotlib import pyplot as plt
from adjustText import adjust_text
import matplotlib.patheffects as pe

fig = plt.figure(figsize=(16, 16))
sns.scatterplot(
    data=plot_df,
    x='x',
    y='y',
    hue='Topic',
    palette=color_key,
    alpha=0.4,
    sizes=(0.4, 10),
    size='Length',
    legend=False,
)

# Annotate top 50 topics
texts, xs, ys = [], [], []
for _, row in mean_df.iterrows():
    topic = int(row['Topic'])
    topic_words = topic_model.get_topic(topic) or []
    name = ' - '.join([word for word, _ in topic_words[:3]])

    if topic <= 50 and name:
        xs.append(row['x'])
        ys.append(row['y'])
        texts.append(plt.text(
            row['x'],
            row['y'],
            name,
            size=10,
            ha='center',
            color=color_key[str(topic)],
            path_effects=[pe.withStroke(linewidth=0.5, foreground='black')],
        ))

adjust_text(texts, x=xs, y=ys, time_lim=1, force_text=(0.01, 0.02), force_static=(0.01, 0.02), force_pull=(0.5, 0.5))
plt.tight_layout()
plt.savefig('topic_scatter.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
from sentence_transformers import SentenceTransformer

In [ ]:
topic_model.reduce_topics(docs, nr_topics=20)

In [ ]:
# Reuse embeddings created during BERTopic fitting. Re-run this only if you intentionally changed docs or the embedding model.
embeddings = sentence_embeddings


In [ ]:
import pickle
with open('sentence_embeddings', 'wb') as file:
    pickle_embeddings = pickle.dump(embeddings, file)


In [ ]:
with open('docs.txt', 'r', encoding='utf-8') as file:
    f = file.read()
f[:1000]


In [ ]:
import pickle

with open('sentence_embeddings', 'rb') as f:
    file = pickle.load(f)
file


In [ ]:
import bertopic

topic_model = BERTopic.load('topic_model_earnings_calls')


# Big Heading!!!

In [ ]:
import pickle
import pandas as pd

# Load saved topic model and sentence embeddings from this project directory.
topic_model = BERTopic.load('topic_model_earnings_calls')
with open('sentence_embeddings', 'rb') as f:
    sentence_embeddings = pickle.load(f)

with open('docs.txt', 'r', encoding='utf-8') as file:
    docs = [doc.strip() for doc in file.read().split('\n\n') if doc.strip()]

print(f'Loaded {len(docs):,} documents and {len(sentence_embeddings):,} embeddings')


In [ ]:
topic_model.get_document_info(docs)

In [ ]:
# Placeholder for sentiment data if you join sentiment outputs later.
# The local topic-modeling workflow does not define a `sentiments` dataframe.
print('No sentiment dataframe is loaded in this notebook.')


In [ ]:
topic_model.get_topic_info()

In [ ]:
topic_model.reduce_topics(docs, nr_topics=30)

In [ ]:
topic_model.get_topic_info()

In [ ]:
topic_model.get_topic(0)

The information that best describes this topic model is that there are two topics that hold the majority of the documents. The first of these is the 'everything else' topic or topic -1. BERTopic uses this topic to hold all document that do not fit cleanly into any of the other topics. The second of these topics is topic zero. This topic holds information related to analyst statements about finances and investments.

It is predictable that the majority of documents would speak over this topic as earnings calls allow companies a platform to inform their shareholders about these topics and allow shareholders to ask questions about these topics. These topics align well with the financial backbone of the company and how it will grow into the future which is likely to affect share price. This is a sign that our model returns results that fit well with prior expectations.

But the new predictions that can come from our model would be information that is important to investors about the state of markets. About the sentiment of topics. About the sentiment about companies. But we don't have the sentiment data. But when I think about what topics may be involved in a corpus like this I want to think about the Covid 19, the ESG, etc.

When we reduce the topics to 30 we see topics that cover market demand, analyst and investor reports, signoffs, finance and accounting, goldman analysts, investor signoffs, empty chunks, vaccines and vaccination of covid, closing remarks, billing, 

What is interesting about our model is that 250 documents can be identified with the topic of Covid 19 vaccination. This highlights the impact that Covid had on the finance world. Earnings calls also identify 33 chunks that discuss security analysis, also highlighting the impact of security on the modern day digital world.

The model also has some predictably common topics, for instance demand in the market is the largest topic with 16565 chunks. Investment analysis as a topic is th esecond largest with 2834 chunks. Accounting analysis after investment analysis. 

Modern technologies like Tesla and electric vehicles are also an important topic in the earnings calls with the 11th largest and 36 chunks with that topic.

Inflation is a suprisingly small topic in the earnings calls. Being the 24th largest topic with only 12 chunks in it.

In [ ]:
topic_model.visualize_topics()

In [ ]:
#sentence_embeddings is embeddings. Could also be reduced. topics is topic model topics but optional.
# hide annotations is false, hide document cover is true, custom labels is false, title is optional

topic_model.visualize_documents(docs, embeddings=sentence_embeddings, hide_document_hover=True)

In [ ]:
reduced_embeddings = UMAP(n_neighbors=10, n_components=2, min_dist=0.0, metric='cosine').fit_transform(sentence_embeddings)

print("Finished UMAP")

topic_model.visualize_documents(docs, reduced_embeddings=reduced_embeddings, hide_document_hover=True, hide_annotations=True)